# SymPy for Quantum Simulation

In [1]:
# Symbolic computation.
import sympy as sp

# Quantum operations.
import sympy.physics.quantum as spq

In [2]:
def is_valid_qubit(state_vector):
    """
    Check if a complex vector represents a valid qubit state.
    A valid qubit must be normalized (i.e., |alpha|^2 + |beta|^2 = 1).
    
    Args:
        state_vector: A SymPy Matrix representing a qubit state [alpha, beta]
    
    Returns:
        bool: True if the state is a valid qubit, False otherwise
    """
    # Check if the vector has exactly two elements
    if state_vector.shape != (2, 1):
        return False
    
    # Calculate the sum of squared magnitudes
    probability_sum = 0
    for element in state_vector:
        probability_sum += (element * sp.conjugate(element))
    
    # Simplify the expression
    probability_sum = sp.simplify(probability_sum)
    
    # For symbolic expressions, we can only check if the simplified expression equals 1
    if probability_sum.is_number:
        return sp.Abs(probability_sum - 1) < 1e-10
    else:
        return probability_sum == 1

In [3]:
def apply_gate(gate, state):
    """
    Apply a quantum gate (unitary matrix) to a quantum state.
    
    Args:
        gate: A SymPy Matrix representing a unitary matrix
        state: A SymPy Matrix representing a quantum state
    
    Returns:
        sympy.Matrix: The resulting quantum state after applying the gate
    """
    # Check if dimensions match
    if gate.shape[1] != state.shape[0]:
        raise ValueError("Gate and state dimensions do not match")
    
    # Apply the gate to the state
    result = gate * state
    
    # Simplify the result
    return sp.simplify(result)

## Common Qubits

In [4]:
# |0⟩
qubit_0 = sp.Matrix([1, 0])

# Show.
qubit_0

Matrix([
[1],
[0]])

In [5]:
# Check if |0⟩ is a valid qubit.
is_valid_qubit(qubit_0)

True

In [6]:
# |1⟩
qubit_1 = sp.Matrix([0, 1])

# Show.
qubit_1

Matrix([
[0],
[1]])

In [7]:
# Check if |1⟩ is a valid qubit.
is_valid_qubit(qubit_1)

True

$\ket{+} = \frac{1}{\sqrt{2}} \ket{0} + \frac{1}{\sqrt{2}} \ket{1}$

In [8]:
# |+⟩
qubit_plus = sp.Matrix([1/sp.sqrt(2), 1/sp.sqrt(2)])

# Show.
qubit_plus

Matrix([
[sqrt(2)/2],
[sqrt(2)/2]])

In [9]:
# Check if |+⟩ is a valid qubit.
is_valid_qubit(qubit_plus)

True

$\ket{-} = \frac{1}{\sqrt{2}} \ket{0} - \frac{1}{\sqrt{2}} \ket{1}$

In [10]:
# |-⟩
qubit_minus = sp.Matrix([1/sp.sqrt(2), -1/sp.sqrt(2)])

In [11]:
# Show.
qubit_minus

Matrix([
[ sqrt(2)/2],
[-sqrt(2)/2]])

In [12]:
# Check if |-⟩ is a valid qubit.
is_valid_qubit(qubit_minus)

True

In [13]:
# (1/√2)|0⟩ + (i/√2)|1⟩.
qubit_complex = sp.Matrix([1/sp.sqrt(2), sp.I/sp.sqrt(2)])

# Show.
qubit_complex

Matrix([
[  sqrt(2)/2],
[sqrt(2)*I/2]])

In [14]:
# Check if the complex qubit is valid.
is_valid_qubit(qubit_complex)

True

## Invalid Qubit States

In [15]:
# Invalid qubit state - not normalized.
invalid_qubit = sp.Matrix([1, 1])

# Show.
invalid_qubit

Matrix([
[1],
[1]])

In [16]:
# Check if the invalid qubit is valid.
is_valid_qubit(invalid_qubit)

False

## Symbollic Qubit States

In [17]:
# Two new complex symbols.
alpha, beta = sp.symbols('alpha beta', complex=True)

In [18]:
# In a qubit.
symbolic_qubit = sp.Matrix([alpha, beta])


In [19]:
# Show.
symbolic_qubit

Matrix([
[alpha],
[ beta]])

In [20]:
# Define constraint that |alpha|^2 + |beta|^2 = 1
constraint = alpha * sp.conjugate(alpha) + beta * sp.conjugate(beta)

# Show.
constraint

alpha*conjugate(alpha) + beta*conjugate(beta)

## Tensor Products

In [21]:
def tensor_product(qubit_list):
    """
    Calculate the tensor product of multiple qubits.
    
    Args:
        qubit_list: A list of SymPy Matrices, each representing a qubit state.
    
    Returns:
        sympy.Matrix: The tensor product of all qubits.
    """
    # Start with the first qubit
    result = qubit_list[0]
    
    # Compute tensor product with remaining qubits
    for qubit in qubit_list[1:]:
        result = spq.tensorproduct.TensorProduct(result, qubit)
    
    return result

In [22]:
# Calculate tensor product of |0⟩ and |1⟩ to get |01⟩.
tensor_01 = tensor_product([qubit_0, qubit_1])

# Show.
print("Tensor product |0⟩ ⊗ |1⟩ = ")
tensor_01


Tensor product |0⟩ ⊗ |1⟩ = 


Matrix([
[0],
[1],
[0],
[0]])

In [23]:
# Calculate tensor product of |+⟩ and |-⟩.
tensor_plus_minus = tensor_product([qubit_plus, qubit_minus])

# Show.
print("Tensor product |+⟩ ⊗ |-⟩ = ")
tensor_plus_minus


Tensor product |+⟩ ⊗ |-⟩ = 


Matrix([
[ 1/2],
[-1/2],
[ 1/2],
[-1/2]])

In [24]:
# Calculate tensor product of three qubits
tensor_three = tensor_product([qubit_0, qubit_plus, qubit_1])

# Show.
print("Tensor product |0⟩ ⊗ |+⟩ ⊗ |1⟩:")
tensor_three


Tensor product |0⟩ ⊗ |+⟩ ⊗ |1⟩:


Matrix([
[        0],
[sqrt(2)/2],
[        0],
[sqrt(2)/2],
[        0],
[        0],
[        0],
[        0]])

## Unitary Matrices

In [25]:
def is_unitary(matrix):
    """
    Check if a matrix is unitary.
    A unitary matrix U satisfies U† U = U U† = I, where U† is the conjugate transpose.
    
    Args:
        matrix: A SymPy Matrix
    
    Returns:
        bool: True if the matrix is unitary, False otherwise
    """
    # Check if the matrix is square
    if matrix.shape[0] != matrix.shape[1]:
        return False
    
    # Calculate conjugate transpose (dagger)
    dagger = matrix.H
    
    # Check if U† U = I
    product = dagger * matrix
    identity = sp.eye(matrix.shape[0])
    
    # Simplify the product
    product = sp.simplify(product)
    
    # Check if product equals identity
    return product == identity

In [26]:
# Identity gate.
I_gate = sp.Matrix([[1, 0], [0, 1]])

# Show.
I_gate

Matrix([
[1, 0],
[0, 1]])

In [27]:
# Pauli-X (NOT) gate.
X_gate = sp.Matrix([[0, 1], [1, 0]])

# Show.
X_gate

Matrix([
[0, 1],
[1, 0]])

In [28]:
# Hadamard gate.
H_gate = sp.Matrix([[1, 1], [1, -1]]) / sp.sqrt(2)

# Show.
H_gate

Matrix([
[sqrt(2)/2,  sqrt(2)/2],
[sqrt(2)/2, -sqrt(2)/2]])

In [29]:
# Define a non-unitary matrix.
non_unitary = sp.Matrix([[1, 1], [1, 1]])

# Show.
non_unitary

Matrix([
[1, 1],
[1, 1]])

In [30]:
# Unitary?
print(f"Is I unitary? {is_unitary(I_gate)}")
print(f"Is X unitary? {is_unitary(X_gate)}")
print(f"Is H unitary? {is_unitary(H_gate)}")
print(f"Is [[1, 1], [1, 1]] unitary? {is_unitary(non_unitary)}")

Is I unitary? True
Is X unitary? True
Is H unitary? True
Is [[1, 1], [1, 1]] unitary? False


In [31]:
# Pauli-Z gate.
Z_gate = sp.Matrix([[1, 0], [0, -1]])

# Pauli-Y gate.
Y_gate = sp.Matrix([[0, -sp.I], [sp.I, 0]])

# Unitary?
print(f"Is Z unitary? {is_unitary(Z_gate)}")
print(f"Is Y unitary? {is_unitary(Y_gate)}")


Is Z unitary? True
Is Y unitary? True


## Applying Gates

In [32]:
# Applying X gate to |0⟩ should give |1⟩.
x_applied = apply_gate(X_gate, qubit_0)

# Show.
print("X|0⟩ = ")
x_applied


X|0⟩ = 


Matrix([
[0],
[1]])

In [33]:
print(f"Is X|0⟩ equal to |1⟩? {x_applied == qubit_1}")

Is X|0⟩ equal to |1⟩? True


In [34]:
# Applying H gate to |0⟩ should give |+⟩.
h_applied = apply_gate(H_gate, qubit_0)

# Show.
print("H|0⟩ = ")
h_applied


H|0⟩ = 


Matrix([
[sqrt(2)/2],
[sqrt(2)/2]])

In [35]:
print(f"Is H|0⟩ equal to |+⟩? {h_applied == qubit_plus}")

Is H|0⟩ equal to |+⟩? True


In [ ]:
# Applying H gate to |1⟩ should give |-⟩.
h_applied_1 = apply_gate(H_gate, qubit_1)

# Show.
print("H|1⟩ = ")
h_applied_1


H|1⟩ = 


Matrix([
[ sqrt(2)/2],
[-sqrt(2)/2]])

In [37]:
print(f"Is H|1⟩ equal to |-⟩? {h_applied_1 == qubit_minus}")

Is H|1⟩ equal to |-⟩? True


## Multi-Qubit Operations

In [38]:
# Create a 2-qubit system |00⟩.
two_qubit_system = tensor_product([qubit_0, qubit_0])

# Show.
print("Initial state |00⟩ = ")
two_qubit_system


Initial state |00⟩ = 


Matrix([
[1],
[0],
[0],
[0]])

In [39]:
# Define a CNOT gate.
CNOT = sp.Matrix([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
])

# Show.
CNOT

Matrix([
[1, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 0, 1],
[0, 0, 1, 0]])

In [40]:
# Apply CNOT to |00⟩.
cnot_applied = apply_gate(CNOT, two_qubit_system)

# Show.
print("CNOT|00⟩ = ")
cnot_applied


CNOT|00⟩ = 


Matrix([
[1],
[0],
[0],
[0]])

In [41]:
# Apply CNOT to |10⟩.
state_10 = tensor_product([qubit_1, qubit_0])
cnot_applied_10 = apply_gate(CNOT, state_10)

# Show.
print("CNOT|10⟩ = ")
cnot_applied_10


CNOT|10⟩ = 


Matrix([
[0],
[0],
[0],
[1]])

In [42]:
# Verify that CNOT|10⟩ = |11⟩.
state_11 = tensor_product([qubit_1, qubit_1])

# Show.
print(f"Is CNOT|10⟩ equal to |11⟩? {cnot_applied_10 == state_11}")


Is CNOT|10⟩ equal to |11⟩? True


## Symbolic Computation

In [ ]:
# Create a symbolic phase gate
theta = sp.symbols('theta', real=True)
phase_gate = sp.Matrix([
    [1, 0],
    [0, sp.exp(sp.I*theta)]
])

# Show.
print("Symbolic phase gate:")
phase_gate


Symbolic phase gate:


Matrix([
[1,            0],
[0, exp(I*theta)]])

In [44]:
# Check if it's unitary (should be True for any theta)
print(f"Is phase gate unitary? {is_unitary(phase_gate)}")

Is phase gate unitary? True


In [ ]:
# Apply to |+⟩ state
phase_applied = apply_gate(phase_gate, qubit_plus)

# Show.
print("Phase(theta)|+⟩ = ")
phase_applied

Phase(theta)|+⟩ = 


Matrix([
[             sqrt(2)/2],
[sqrt(2)*exp(I*theta)/2]])

In [ ]:
# Substitute a specific value for theta (e.g., π/2).
phase_applied_pi_2 = phase_applied.subs(theta, sp.pi/2)

# Show.
print("Phase(π/2)|+⟩ = ")
phase_applied_pi_2

Phase(π/2)|+⟩ = 


Matrix([
[  sqrt(2)/2],
[sqrt(2)*I/2]])

## End